# v01 — EDA & Indikator: XAUUSD M5 (+ H1 sebagai konteks)

**Tujuan versi ini:**
1. Gabungkan seluruh CSV mentah XAUUSD M5 & H1 (2019 - Agu 2026) dari `dataset/raw/`
2. Validasi kualitas data (gap waktu, duplikat, anomali OHLC, null)
3. Hitung semua indikator (`app.utils.indicators.add_all_indicators`)
4. Simpan dataset gabungan + indikator ke `dataset/processed/m5_scalping/v01/`

**Log ringkas hasil EDA (dari eksplorasi awal):** 16 file CSV (8 tahun x 2 timeframe), 0 duplikat,
0 null, 0 anomali OHLC, 0 volume/harga negatif, tidak ada overlap antar file tahun. Gap waktu yang
terdeteksi (~250-260/tahun) semuanya adalah weekend, jeda harian broker (~1 jam), atau hari libur —
bukan data hilang. Detail dieksekusi ulang di notebook ini supaya jadi log yang bisa diverifikasi ulang.

**Konvensi folder output:** `dataset/processed/<nama_strategi>/<versi>/` — dipisah per strategi dulu
(mis. `m5_scalping`, nanti `m15_scalping`, `h1_swing`, dst), baru di dalamnya per versi (`v01`, `v02`, ...).
Ini supaya jelas output milik strategi/timeframe mana, tidak tercampur kalau ada riset lain berjalan
paralel.</cell id="ed154dc9">


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import glob

import pandas as pd

from app.utils.indicators import add_all_indicators

STRATEGY_NAME = "m5_scalping"
VERSION = "v01"

RAW_DIR = PROJECT_ROOT / "dataset" / "raw"
OUT_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME / VERSION
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 140)

## 1. Load & gabungkan semua file per timeframe

In [2]:
def load_and_concat(pattern: str) -> pd.DataFrame:
    files = sorted(glob.glob(str(RAW_DIR / pattern)))
    frames = []
    for f in files:
        df = pd.read_csv(f)
        df["datetime"] = pd.to_datetime(df["datetime"])
        frames.append(df)
    combined = pd.concat(frames, ignore_index=True)
    combined = combined.sort_values("datetime").drop_duplicates(subset="datetime").reset_index(drop=True)
    return combined


df_m5 = load_and_concat("xauusd_m5_*.csv")
df_h1 = load_and_concat("xauusd_h1_*.csv")

print("M5:", df_m5.shape, df_m5["datetime"].min(), "->", df_m5["datetime"].max())
print("H1:", df_h1.shape, df_h1["datetime"].min(), "->", df_h1["datetime"].max())

M5: (518403, 7) 2019-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00
H1: (44809, 7) 2019-01-01 23:00:00+00:00 -> 2026-08-06 11:00:00+00:00


## 2. Validasi kualitas data

In [3]:
def data_quality_report(df: pd.DataFrame, expected_freq_min: int) -> dict:
    n = len(df)
    dup_ts = df["datetime"].duplicated().sum()
    nulls = df.isna().sum().sum()
    bad_hl = (df["high"] < df["low"]).sum()
    bad_hc = (df["high"] < df[["open", "close"]].max(axis=1)).sum()
    bad_lc = (df["low"] > df[["open", "close"]].min(axis=1)).sum()
    zero_vol = (df["volume"] == 0).sum()
    neg_price = (df[["open", "high", "low", "close"]] <= 0).any(axis=1).sum()

    diffs = df["datetime"].diff().dropna()
    expected = pd.Timedelta(minutes=expected_freq_min)
    gaps = diffs[diffs > expected]

    return {
        "rows": n,
        "dup_timestamps": dup_ts,
        "nulls": nulls,
        "bad_high<low": bad_hl,
        "bad_high<max(o,c)": bad_hc,
        "bad_low>min(o,c)": bad_lc,
        "zero_volume": zero_vol,
        "neg_or_zero_price": neg_price,
        "n_gaps": len(gaps),
        "max_gap": gaps.max() if len(gaps) else pd.Timedelta(0),
        "gaps_over_1day": int((gaps > pd.Timedelta(days=1)).sum()),
    }


report_m5 = data_quality_report(df_m5, expected_freq_min=5)
report_h1 = data_quality_report(df_h1, expected_freq_min=60)
pd.DataFrame([{"timeframe": "M5", **report_m5}, {"timeframe": "H1", **report_h1}])

,timeframe,rows,dup_timestamps,nulls,bad_high<low,"bad_high<max(o,c)","bad_low>min(o,c)",zero_volume,neg_or_zero_price,n_gaps,max_gap,gaps_over_1day
0,M5,518403,0,0,0,0,0,0,0,1960,6 days 23:05:00,443
1,H1,44809,0,0,0,0,0,0,0,1962,4 days 00:00:00,402


**Interpretasi gap:** gap-gap yang muncul (mayoritas ~1 jam harian + ~3 hari weekend) adalah pola pasar
normal (jeda harian broker + market tutup Sabtu-Minggu + hari libur), bukan indikasi data hilang.
Kalau `gaps_over_1day` jauh lebih besar dari jumlah minggu dalam rentang data, baru perlu dicurigai.

In [4]:
# Cek range harga per tahun — deteksi anomali skala harga (mis. kesalahan unit/split)
df_m5["year"] = df_m5["datetime"].dt.year
df_m5.groupby("year")["close"].agg(["min", "max", "mean"])

,min,max,mean
year,,,
2019,1266.675,1556.358,1393.051104
2020,1453.745,2071.315,1771.384013
2021,1677.458,1957.775,1798.896432
2022,1615.728,2069.358,1801.310164
2023,1806.855,2138.405,1943.479843
2024,2001.995,2789.578,2402.714758
2025,2618.498,4548.365,3465.700217
2026,3957.235,5587.628,4588.166375


## 3. Hitung semua indikator

In [5]:
df_m5 = df_m5.drop(columns=["year"])
df_m5_ind = add_all_indicators(df_m5)
df_h1_ind = add_all_indicators(df_h1)

print("M5 + indikator:", df_m5_ind.shape)
print("H1 + indikator:", df_h1_ind.shape)
df_m5_ind.tail()

M5 + indikator: (518403, 75)
H1 + indikator: (44809, 75)


,timestamp,datetime,open,high,low,close,volume,ema_9,ema_20,ema_50,...,ob_bull,ob_bear,bos_bull,bos_bear,choch_bull,choch_bear,liq_bull_sweep,liq_bear_sweep,candle_pat,candle_ex
518398,1786018500000,2026-08-06 12:15:00+00:00,4256.085,4257.275,4253.035,4256.925,0.1179,4259.534329,4262.593394,4265.683222,...,0,0,0,0,0,0,1,0,1,1
518399,1786018800000,2026-08-06 12:20:00+00:00,4256.925,4262.915,4255.875,4260.645,0.1466,4259.756463,4262.407833,4265.485645,...,0,0,0,0,0,0,0,0,0,0
518400,1786019100000,2026-08-06 12:25:00+00:00,4260.685,4261.635,4256.535,4261.595,0.0804,4260.124171,4262.330420,4265.333070,...,0,0,0,0,0,0,0,0,1,2
518401,1786019400000,2026-08-06 12:30:00+00:00,4261.885,4269.755,4257.925,4267.565,0.1259,4261.612337,4262.828952,4265.420597,...,0,0,0,0,0,0,0,0,0,2
518402,1786019700000,2026-08-06 12:35:00+00:00,4267.805,4270.565,4266.935,4269.235,0.1002,4263.136869,4263.439052,4265.570181,...,0,0,0,0,0,0,0,0,0,2


In [6]:
# Cek NaN per kolom (wajar tinggi di awal karena warm-up period indikator dgn window besar, mis. sma_200 & fibonacci)
df_m5_ind.isna().sum().sort_values(ascending=False).head(15)

sma_200           199
ichi_span_b        77
ichi_span_a        51
ichi_cloud_bot     51
ichi_cloud_top     51
sma_50             49
fib_swing_low      49
fib_swing_high     49
fib_500            49
fib_236            49
fib_786            49
fib_382            49
fib_618            49
ichi_kijun         25
vwap               19
dtype: int64

## 4. Simpan dataset gabungan + indikator ke `dataset/processed/m5_scalping/v01/`

In [7]:
df_m5_ind.to_csv(OUT_DIR / "xauusd_m5_full_indicators.csv", index=False)
df_h1_ind.to_csv(OUT_DIR / "xauusd_h1_full_indicators.csv", index=False)

print("Tersimpan ke:", OUT_DIR)
for f in sorted(OUT_DIR.glob("*.csv")):
    print(" -", f.name, f"({f.stat().st_size / 1e6:.1f} MB)")

Tersimpan ke: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v01
 - xauusd_h1_full_indicators.csv (37.6 MB)
 - xauusd_m5_full_indicators.csv (435.9 MB)
